# Computer Graphics — CO1

**2310218L.CO.1** — *Apply OpenGL graphics primitives to develop graphics applications.*
**[L3]**

This notebook covers **CO1 only**, in depth. The other outcomes are in
[`subjects/03-computer-graphics.md`](subjects/03-computer-graphics.md).

---

## What a graphics primitive actually is

A screen is a grid of dots. Nothing on it is a line, a box or a circle — those are ideas
we impose. A **primitive** is the smallest shape a system knows how to draw, and every
picture is assembled from them.

The chain is always the same:

```
picture  ->  primitives  ->  pixels  ->  the screen
```

Our primitives are the rectangle, the line, the circle and text. Each one knows how to turn
itself into pixels, and the application is built by composing them.

---

## The surface everything is drawn on

Before any primitive can exist, there has to be somewhere to put pixels.

**What this does.** Holds the drawing surface: a width, a height, and one array of pixels.
Nothing in the project writes to the screen directly; everything writes here first.

```cpp
class Canvas {
    int    width_;
    int    height_;
    Pixel* buffer_;
};
```
<sub>src/ui/Canvas.h — members only, methods omitted</sub>

**What this does.** Turns an (x, y) position into a place in that array. This is the
single function every primitive in the project eventually calls.

```cpp
void Canvas::setPixel(int x, int y, Pixel p) {
    if (inBounds(x, y)) buffer_[y * width_ + x] = p;
}
```
<sub>src/ui/Canvas.cpp:41</sub>

**Why this matters more than it looks.** Because every primitive goes through `setPixel`,
the whole drawing system has exactly one place where a pixel is written. Swapping the
console for a graphics window means rewriting that one function — and not a single line of
any primitive above it.

---

## The four primitives

![the four primitives, composed into the diagram](diagrams/cg-primitives.png)

| Primitive | Drawn by | What it draws in our application |
|---|---|---|
| Rectangle | `drawBox`, `drawRect` | the register file, control unit, memory and ALU blocks |
| Line | `drawLineBresenham` | the data bus, and every wire dropping onto it |
| Circle | `drawCircleBresenham` | the junction dots, and the value travelling the bus |
| Text | `drawText` | labels, register contents, ALU operands and results |

### Rectangle — the component blocks

**What this does.** Draws a labelled box. It is built from four lines and four corner
characters, with the title written into the top edge — so even the rectangle is not a
primitive underneath, but a composition of lines.

```cpp
void Canvas::drawBox(int x, int y, int w, int h, const std::string& title) {
    if (w < 2 || h < 2) return;

    for (int i = x + 1; i < x + w - 1; ++i) {
        setPixel(i, y,         '-');
        setPixel(i, y + h - 1, '-');
    }
    for (int j = y + 1; j < y + h - 1; ++j) {
        setPixel(x,         j, '|');
        setPixel(x + w - 1, j, '|');
    }
```
<sub>src/ui/Canvas.cpp:331 — first half; corners and title follow</sub>

### Text — labels and live values

**What this does.** Writes a string one character at a time, each into its own pixel
position. Text is a primitive here in the same sense as a line: something the canvas knows
how to place directly.

```cpp
void Canvas::drawText(int x, int y, const std::string& text) {
    for (size_t i = 0; i < text.size(); ++i)
        setPixel(x + static_cast<int>(i), y, text[i]);
}
```
<sub>src/ui/Canvas.cpp:326</sub>

---

## Applying them — building the application

CO1 is not about having primitives. It is about **using them to develop a graphics
application**. This is where that happens.

**What this does.** Draws the four component blocks of the processor. Four calls, four
rectangles — this is the composition step, where primitives stop being shapes and start
being a picture of a machine.

```cpp
c.drawBox(1,  1, 24, 6, "REGISTER FILE");
c.drawBox(29, 1, 20, 6, "CONTROL UNIT");
c.drawBox(52, 1, 22, 6, "MEMORY");
c.drawBox(20, 13, 24, 7, "ALU");
```
<sub>src/ui/ConsoleView.cpp:66</sub>

**What this does.** Draws the shared data bus as one long line. The value passed for the
pixel is chosen from the machine's current state — so the same call renders an idle bus or
a live one.

```cpp
Pixel busPix = sig.busActive ? PX_ACTIVE : PX_WIRE;
...
c.drawLineBresenham(2, busY, W - 3, busY, busPix);
```
<sub>src/ui/ConsoleView.cpp:63 and :103</sub>

**This is the idea worth explaining.** The picture is not a fixed drawing. It is redrawn
from machine state on every step, and the primitives take their appearance from that
state. A wire carrying data this cycle and an idle wire are the *same* drawing call with a
different pixel — which is what makes the datapath appear to light up as the program runs.

**What this does.** Draws the junction dots where each block taps onto the bus, again
choosing the character from whether that block is active.

```cpp
c.setPixel(12, busY, regPix == PX_ACTIVE ? 'O' : 'o');
c.setPixel(39, busY, 'O');
c.setPixel(63, busY, memPix == PX_ACTIVE ? 'O' : 'o');
```
<sub>src/ui/ConsoleView.cpp:120</sub>

---

## Getting it onto the screen

**What this does.** Walks the buffer row by row and prints it. Trailing blanks are trimmed
so the output stays tidy. This is the only function that knows the destination is a
terminal.

```cpp
void Canvas::render(std::ostream& os) const {
    for (int y = 0; y < height_; ++y) {
        int last = -1;
        for (int x = 0; x < width_; ++x)
            if (buffer_[y * width_ + x] != PX_EMPTY) last = x;
        for (int x = 0; x <= last; ++x)
            os << buffer_[y * width_ + x];
        os << '\n';
    }
}
```
<sub>src/ui/Canvas.cpp:354</sub>

Replace this one function with something that pushes the buffer to an OpenGL texture, and
the entire application above it is unchanged. That is the payoff of routing everything
through one surface.

---

## How to explain CO1 in one minute

1. **A screen is only dots.** A primitive is the smallest shape a system can draw, and
   every picture is assembled from them.
2. **Our four primitives** are the rectangle, line, circle and text — the same role
   OpenGL's points, lines and triangles play.
3. **They all go through one function,** `setPixel`, which turns an (x, y) into a place in
   one array. That is the whole drawing system's single point of contact with memory.
4. **The application is the composition** — four boxes, a bus, wires and junctions,
   assembled into a picture of a processor.
5. **And it is live.** The picture is redrawn from machine state each step, with the same
   call rendering an idle wire or an active one, which is what makes the datapath light up.